In [1]:
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np

from nRC_parametrization import SOC_func
from nRC_parametrization import SOC_to_OCV
from nRC_parametrization import simulate_battery_model
from nRC_parametrization import RMSE

In [2]:
current_dir = Path.cwd()

parameters_path = current_dir.parent / "validation" / "parameters_for_validation.csv"

static_parameters_dir = current_dir.parent.parent / "Parameters" / "static_parameters"

profiles_dir = current_dir.parent.parent / "Data_preprocessing" / "profiles"

df_parameters = pd.read_csv(parameters_path)


In [3]:
df_parameters

,profile,Battery_ID,SOH,T,SOC,Q_Ah,eta,OCV_file_name,R0,R,tau
0,NEDC,1,94,25,55-35,3.289069,0.993163,OCV_1_+25.csv,0.040102,[ 0.18500965 -0.15799339],[74.79622419 74.46096847]
1,NEDC,1,94,25,75-55,3.289069,0.993163,OCV_1_+25.csv,0.043565,[ 2.38419122 -2.33588565],[76.6982802 75.91917751]
2,NEDC,1,94,25,95-75,3.289069,0.993163,OCV_1_+25.csv,0.044178,[-3.1062521 3.13802852],[76.12695194 76.00372429]
3,UDDS,1,94,25,55-35,3.289069,0.993163,OCV_1_+25.csv,0.038852,[ 0.05812656 -0.0271138 ],[70.43697862 64.5548939 ]
4,UDDS,1,94,25,75-55,3.289069,0.993163,OCV_1_+25.csv,0.040251,[ 0.14614713 -0.09671487],[70.3419654 63.7584806]
...,...,...,...,...,...,...,...,...,...,...,...
265,impulse_288_s,5,98,35,75-55,3.434987,0.991690,OCV_5_+35.csv,0.041669,[ 0.11122546 -0.09116178],[63.50124231 51.079858 ]
266,impulse_288_s,5,98,35,95-75,3.434987,0.991690,OCV_5_+35.csv,0.037150,[ 0.04318691 -0.02255343],[60.161704 49.73728641]
267,impulse_72_s,5,98,35,55-35,3.434987,0.991690,OCV_5_+35.csv,0.037404,[ 0.0295919 -0.01634364],[23.43636714 5.62030503]
268,impulse_72_s,5,98,35,75-55,3.434987,0.991690,OCV_5_+35.csv,0.038704,[ 0.0609155 -0.0486512],[15.63131377 6.06766978]


In [4]:
df_profiles_description =  pd.read_csv(static_parameters_dir / "profiles_description_static_parameters.csv")

In [5]:
df_profiles_description

,bat_num,temp,profile,U_start_V,U_end_V,file_name,Q_Ah,eta,OCV_file_name,SOC_start,SOC_end,SOH
0,1,25,impulse_72_s,4.11093,4.10372,1_+25_impulse_72_s_01.csv,3.289069,0.993163,OCV_1_+25.csv,96.579538,95.849146,93.973411
1,1,25,impulse_144_s,4.10372,4.09365,1_+25_impulse_144_s_02.csv,3.289069,0.993163,OCV_1_+25.csv,95.849146,94.516093,93.973411
2,1,25,impulse_288_s,4.09365,4.07940,1_+25_impulse_288_s_03.csv,3.289069,0.993163,OCV_1_+25.csv,94.516093,91.610379,93.973411
3,1,25,impulse_72_s,4.07940,4.07487,1_+25_impulse_72_s_04.csv,3.289069,0.993163,OCV_1_+25.csv,91.610379,90.357663,93.973411
4,1,25,impulse_144_s,4.07487,4.06139,1_+25_impulse_144_s_05.csv,3.289069,0.993163,OCV_1_+25.csv,90.357663,86.572312,93.973411
...,...,...,...,...,...,...,...,...,...,...,...,...
535,5,35,NEDC,3.95779,3.75069,5_+35_NEDC_02.csv,3.434987,0.991690,OCV_5_+35.csv,76.178929,55.833428,98.142481
536,5,35,NEDC,3.75069,3.59529,5_+35_NEDC_03.csv,3.434987,0.991690,OCV_5_+35.csv,55.833428,36.393882,98.142481
537,5,35,WLTC,4.11297,3.90308,5_+35_WLTC_01.csv,3.434987,0.991690,OCV_5_+35.csv,96.318219,70.142435,98.142481
538,5,35,WLTC,3.90308,3.68861,5_+35_WLTC_02.csv,3.434987,0.991690,OCV_5_+35.csv,70.142435,49.435377,98.142481


In [6]:
profiles_for_validation = ["UDDS", "NEDC", "WLTC"]

SOC_ranges = {
    "95-75": "01",
    "75-55": "02",
    "55-35": "03"
}

df_validation = df_parameters.copy()

for p in ["UDDS", "NEDC", "WLTC"]:
    df_validation[f"{p}_rmse"] = np.nan


for idx, row in df_parameters.iterrows():
    battery_id = row["Battery_ID"]
    temp = row["T"]
    soc_range = row["SOC"]
    Q_Ah = row["Q_Ah"]
    eta = row["eta"]

    df_OCV = pd.read_csv(static_parameters_dir / row["OCV_file_name"])

    for profile_name in profiles_for_validation:
        profile_file_name = f'{battery_id}_+{temp}_{profile_name}_{SOC_ranges[soc_range]}.csv'
        
        df_profile = pd.read_csv(profiles_dir / profile_file_name)

        
        
        z0 = df_profiles_description.loc[df_profiles_description["file_name"] == profile_file_name, "SOC_start"].iloc[0]
        
        t = df_profile["t,s"].values
        I = df_profile["I,A"].values
        U = df_profile["U,V"].values
        
        SOC = SOC_func(I, t, z0, Q_Ah, eta)
        
        OCV = SOC_to_OCV(SOC, df_OCV)

        R0 = float(row["R0"])
        R = np.fromstring(row["R"].strip("[]"), sep=" ")
        tau = np.fromstring(row["tau"].strip("[]"), sep=" ")
        
        dynamic_parameters = {'R0': R0, 'R': R, 'tau': tau}
        
        U_sim = simulate_battery_model(I, t, OCV, eta, Q_Ah, dynamic_parameters)
        
        U_rmse = RMSE(U, U_sim)

        df_validation.loc[idx, f"{profile_name}_rmse"] = U_rmse

        print(idx)

        
        

0
0
0
1
1
1
2
2
2
3
3
3
4
4
4
5
5
5
6
6
6
7
7
7
8
8
8
9
9
9
10
10
10
11
11
11
12
12
12
13
13
13
14
14
14
15
15
15
16
16
16
17
17
17
18
18
18
19
19
19
20
20
20
21
21
21
22
22
22
23
23
23
24
24
24
25
25
25
26
26
26
27
27
27
28
28
28
29
29
29
30
30
30
31
31
31
32
32
32
33
33
33
34
34
34
35
35
35
36
36
36
37
37
37
38
38
38
39
39
39
40
40
40
41
41
41
42
42
42
43
43
43
44
44
44
45
45
45
46
46
46
47
47
47
48
48
48
49
49
49
50
50
50
51
51
51
52
52
52
53
53
53
54
54
54
55
55
55
56
56
56
57
57
57
58
58
58
59
59
59
60
60
60
61
61
61
62
62
62
63
63
63
64
64
64
65
65
65
66
66
66
67
67
67
68
68
68
69
69
69
70
70
70
71
71
71
72
72
72
73
73
73
74
74
74
75
75
75
76
76
76
77
77
77
78
78
78
79
79
79
80
80
80
81
81
81
82
82
82
83
83
83
84
84
84
85
85
85
86
86
86
87
87
87
88
88
88
89
89
89
90
90
90
91
91
91
92
92
92
93
93
93
94
94
94
95
95
95
96
96
96
97
97
97
98
98
98
99
99
99
100
100
100
101
101
101
102
102
102
103
103
103
104
104
104
105
105
105
106
106
106
107
107
107
108
108
108
109
109
109
110
110
11

In [7]:
df_validation[["UDDS_rmse", "NEDC_rmse", "WLTC_rmse"]].describe()

,UDDS_rmse,NEDC_rmse,WLTC_rmse
count,270.000000,270.000000,270.000000
mean,0.006253,0.007084,0.010511
std,0.004322,0.005031,0.006616
min,0.000858,0.000875,0.001549
25%,0.003505,0.003876,0.006339
50%,0.004803,0.005692,0.008714
75%,0.008700,0.009589,0.014119
max,0.026015,0.035705,0.044176


In [8]:
df_validation.to_csv(current_dir.parent / "validation" / "validation_results.csv", index=False)